# Recommendation system using ALS 
Alternating Least Squares

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('rec').getOrCreate()

In [3]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

In [5]:
data = spark.read.csv('movielens_ratings.csv',inferSchema=True,header=True)

In [6]:
data.head()

Row(movieId=2, rating=3.0, userId=0)

In [7]:
data.describe().show()

+-------+------------------+------------------+------------------+
|summary|           movieId|            rating|            userId|
+-------+------------------+------------------+------------------+
|  count|              1501|              1501|              1501|
|   mean| 49.40572951365756|1.7741505662891406|14.383744170552964|
| stddev|28.937034065088994| 1.187276166124803| 8.591040424293272|
|    min|                 0|               1.0|                 0|
|    max|                99|               5.0|                29|
+-------+------------------+------------------+------------------+



In [9]:
(training, test) = data.randomSplit([0.8, 0.2])

In [10]:
als = ALS(maxIter=5, regParam=0.01, userCol="userId", itemCol="movieId", ratingCol="rating")
model = als.fit(training)

In [11]:
predictions = model.transform(test)

In [12]:
predictions.show()

+-------+------+------+-----------+
|movieId|rating|userId| prediction|
+-------+------+------+-----------+
|      0|   3.0|    28|   2.882566|
|      1|   1.0|    26|-0.58770555|
|      0|   1.0|    22|  1.4367697|
|      5|   1.0|    13|  1.4498708|
|      0|   1.0|     3| 0.71759486|
|      0|   1.0|    20|  0.6260228|
|      1|   1.0|    20|  1.9896951|
|      4|   2.0|    20|  1.4523528|
|      0|   1.0|    19|  0.9859377|
|      4|   1.0|    19|  0.2574209|
|      3|   1.0|     9|  0.1920007|
|      4|   2.0|     8|  3.3097444|
|      1|   1.0|     7|-0.76238453|
|      2|   2.0|     7|-0.27274877|
|      3|   1.0|     7|  2.1182418|
|      4|   1.0|     7|  2.1081855|
|      2|   1.0|    25| 0.63139224|
|      3|   1.0|    29|  1.3596394|
|      5|   1.0|    14|  1.1967658|
|      3|   1.0|     0|  1.2956549|
+-------+------+------+-----------+
only showing top 20 rows



In [13]:
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))

Root-mean-square error = 1.7660847686372296


In [18]:
single_user = test.filter(test['userId']==28).select(['movieId','userId'])

In [19]:
single_user.show()

+-------+------+
|movieId|userId|
+-------+------+
|      0|    28|
|     12|    28|
|     17|    28|
|     24|    28|
|     29|    28|
|     34|    28|
|     36|    28|
|     38|    28|
|     59|    28|
|     63|    28|
|     92|    28|
|     99|    28|
+-------+------+



In [20]:
reccomendations = model.transform(single_user)

In [21]:
reccomendations.orderBy('prediction',ascending=False).show()

+-------+------+----------+
|movieId|userId|prediction|
+-------+------+----------+
|     34|    28|  3.563321|
|     29|    28| 3.5014143|
|     92|    28| 3.4417324|
|      0|    28|  2.882566|
|     12|    28| 1.7626833|
|     24|    28| 1.7428396|
|     36|    28| 1.6080899|
|     38|    28| 1.5028473|
|     99|    28| 1.0471945|
|     59|    28| 1.0410511|
|     17|    28| -0.099318|
|     63|    28|-2.5154767|
+-------+------+----------+

